# AI Data Analyst Agent — Complete Demo
This notebook demonstrates the full workflow: multilingual file loading, data profiling, semantic column matching, statistical analyses, multiple chart types, and an optional live agent call.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from app.agent.orchestrator import run_agent
from app.loaders.pandas_loader import load_data
from app.profiling.profiler import profile_dataset
from app.tools.analysis import analyze_data, resolve_column
from app.tools.visualization import create_chart

## 2. Load a dataset
The loader detects CSV delimiters and encodings and also supports Excel and JSON files.

In [ ]:
samples = sorted((ROOT / 'uploads').glob('*'), key=lambda path: path.stat().st_size, reverse=True)
if not samples:
    raise FileNotFoundError('Place a CSV or Excel file in the uploads directory.')

data_path = samples[0]
df = load_data(data_path)
print(data_path.name, df.shape)
df.head()

## 3. Quick profile and semantic column matching

In [ ]:
profile_dataset(df)

In [ ]:
# Persian aliases are written as Unicode escapes to keep source labels in English.
for concept in ('\u0633\u0646', '\u0641\u0634\u0627\u0631 \u062e\u0648\u0646', 'bmi'):
    try:
        print(f'{concept} → {resolve_column(df, concept)}')
    except ValueError as error:
        print(error)

## 4. Statistical analyses
Available methods: `describe`, `missing`, `correlation`, `outliers`, `frequency`, `group`, and `trend`.

In [ ]:
numeric = df.select_dtypes('number').columns.tolist()
for method in ('describe', 'missing', 'outliers'):
    result = analyze_data(df, method, numeric[:3] or None)
    print(f'\n--- {method} ---')
    print(result)

if len(numeric) >= 2:
    analyze_data(df, 'correlation', numeric[:4])

## 5. Chart gallery
Available charts: histogram, box, scatter, line, bar, pie, and correlation. Persian labels are shaped before Matplotlib saves each figure.

In [ ]:
from IPython.display import Image, display

charts = []
if numeric:
    charts += [create_chart(df, 'histogram', x=numeric[0]), create_chart(df, 'box', x=numeric[0])]
if len(numeric) >= 2:
    charts += [create_chart(df, 'scatter', x=numeric[0], y=numeric[1]), create_chart(df, 'correlation')]

for chart in charts:
    print(chart['title'])
    display(Image(chart['path']))

## 6. Optional live multi-step agent
After configuring `OPENROUTER_API_KEY`, set the value below to `True`. The agent completes every part of the request and returns Markdown in the user's language.

In [ ]:
RUN_LIVE_AGENT = False

if RUN_LIVE_AGENT:
    response = run_agent(df, 'Describe age, detect outliers, and create a histogram and a box plot.')
    print(response.answer)
    print([chart.path for chart in response.charts])
else:
    print('Live execution is disabled to avoid consuming API quota.')

## 7. Web demo
Run `streamlit run streamlit_app.py` from the project root. The same file is the Streamlit Community Cloud entry point and needs no separate backend.